step 1: Install

In [2]:
!pip install -q sentence-transformers

step 2: Embed 5 sentences

In [3]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "The cat is sleeping on the sofa.",       # 0
    "A kitten is napping on the couch.",      # 1 (similar to 0, different words)
    "I love playing cricket on weekends.",    # 2
    "Machine learning models learn from data.", # 3
    "AI systems are trained using datasets.",  # 4 (similar to 3)
]

embeddings = model.encode(sentences)
print(embeddings.shape)   # (5, 384) -> each sentence is 384 numbers

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(5, 384)


step 3: Compare pairs

In [4]:
# similarity matrix (cosine similarity: 1 = same meaning, 0 = unrelated)
sim = util.cos_sim(embeddings, embeddings).numpy()
df = pd.DataFrame(sim.round(2), index=range(5), columns=range(5))
print(df)

# the specific pairs
print("0 vs 1:", util.cos_sim(embeddings[0], embeddings[1]).item())
print("3 vs 4:", util.cos_sim(embeddings[3], embeddings[4]).item())
print("0 vs 2:", util.cos_sim(embeddings[0], embeddings[2]).item())

# cosine DISTANCE = 1 - similarity
print("Distance 0 vs 1:", 1 - util.cos_sim(embeddings[0], embeddings[1]).item())

      0     1     2     3     4
0  1.00  0.73 -0.01 -0.03 -0.04
1  0.73  1.00 -0.07 -0.07 -0.07
2 -0.01 -0.07  1.00  0.03  0.00
3 -0.03 -0.07  0.03  1.00  0.63
4 -0.04 -0.07  0.00  0.63  1.00
0 vs 1: 0.7270480394363403
3 vs 4: 0.6296901702880859
0 vs 2: -0.007622759789228439
Distance 0 vs 1: 0.27295196056365967


## Reflection: What Surprised Me

**Similarity scores**
- Sentences 0 & 1 (cat on sofa vs kitten on couch): 0.727 (distance 0.273)
- Sentences 3 & 4 (machine learning vs AI systems): 0.630
- Sentences 0 & 2 (cat vs cricket): -0.008 (essentially unrelated)

**What matched my intuition:**
The two similar pairs (0-1 and 3-4) scored highest, and the cricket sentence scored close to zero with everything else, exactly as I expected.

**What surprised me:**
1. Sentences 0 and 1 share almost no words, yet scored 0.73. The model understood that "cat" is like "kitten" and "sofa" is like "couch". Bag-of-words from Day 3 would have treated them as unrelated.
2. Some scores were slightly negative (like -0.07). I assumed similarity would always be between 0 and 1, but cosine similarity can go below zero, meaning the vectors point in slightly opposite directions.
3. The cat pair (0.73) scored higher than the ML pair (0.63), even though both pairs mean the same thing. This shows that similarity is a spectrum, not a yes/no answer.

**Key takeaway:**
Embeddings capture meaning, not just word counts. This is the foundation of semantic search, recommendation systems, and RAG.